# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faith-amanze/content-refresh-prioritization/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one content page** (`content_id`), pseudonymized, belonging to one of 32
pseudonymized clients (`client_id`). Every metric column is a **trailing 90-day aggregate**
ending at one shared export time — this file is a single snapshot, not a time series. Two
30-day sub-windows (`*_last_30d`, `*_prev_30d`) sit inside that 90-day window and are used only
to derive `trend_pct`/`trend_direction` — they are NOT a second independent time period.

In [1]:
# ── Grain + coverage check ──
import pandas as pd
import numpy as np
import os

_candidates = [
    "/workspaces/assignment1/data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
]
_data_path = next(p for p in _candidates if os.path.exists(p))
df = pd.read_csv(_data_path)

dup_content_ids = df.groupby("content_id").size().gt(1).sum()
print(f"Rows: {len(df):,} | Columns: {df.shape[1]} | Distinct clients: {df['client_id'].nunique()}")
print(f"content_id values that repeat: {dup_content_ids} (must be 0 for one-row-per-page grain)")
print(f"content_age_days range: {df['content_age_days'].min()}-{df['content_age_days'].max()} "
      f"(every row already >= 90 days old, per the data dictionary)")


Rows: 30,000 | Columns: 44 | Distinct clients: 32
content_id values that repeat: 0 (must be 0 for one-row-per-page grain)
content_age_days range: 90-564 (every row already >= 90 days old, per the data dictionary)


## 2. Fields: feature / label / context / excluded

Every field I might touch, sorted into exactly one bucket, with a one-line why for context and
excluded fields. `trend_direction`/`trend_pct` are the label family; the `*_last_30d`/`*_prev_30d`
columns are excluded here too, ahead of the formal leakage hunt in `w03_feature_leakage_check.ipynb`,
because the label is defined directly from them.

In [2]:
# ── Field classification table ──
field_contract = [
    # column, bucket, why
    ("content_id", "Context", "Pseudonym. Row identifier only, never a feature."),
    ("client_id", "Context", "Pseudonym. Grouping/splitting key only (client-holdout split)."),
    ("search_volume", "Feature", "Describes the target keyword, known before any refresh decision."),
    ("competition", "Feature", "Keyword competition score, known ahead of time."),
    ("competition_level", "Feature", "Categorical version of competition."),
    ("cpc", "Feature", "Keyword economics, known ahead of time."),
    ("content_type", "Feature", "Structural property of the page."),
    ("main_intent", "Feature", "Search-intent category of the page."),
    ("word_count", "Feature", "Content property, known before any performance window."),
    ("char_count", "Feature", "Content property, known before any performance window."),
    ("provider_used", "Excluded", "Data dictionary marks this explicitly 'not a model feature'."),
    ("model_used", "Excluded", "Data dictionary marks this explicitly 'not a model feature'."),
    ("content_age_days", "Feature", "Known at prediction time."),
    ("age_tier", "Excluded", "Redundant restatement of content_age_days."),
    ("age_tier_order", "Excluded", "Redundant restatement of content_age_days."),
    ("days_since_last_update", "Feature", "Known at prediction time; core staleness signal."),
    ("freshness_tier", "Excluded", "Redundant restatement of days_since_last_update."),
    ("word_count_tier", "Excluded", "Redundant restatement of word_count."),
    ("char_count_tier", "Excluded", "Redundant restatement of char_count."),
    ("impressions_90d", "Feature", "Trailing 90-day total, describes current state."),
    ("clicks_90d", "Feature", "Trailing 90-day total."),
    ("pageviews_90d", "Feature", "Trailing 90-day total."),
    ("sessions_90d", "Feature", "Trailing 90-day total."),
    ("users_90d", "Feature", "Trailing 90-day total."),
    ("engaged_sessions_90d", "Feature", "Trailing 90-day total."),
    ("ai_sessions_90d", "Feature", "Trailing 90-day total."),
    ("scroll_events_90d", "Feature", "Trailing 90-day total."),
    ("days_with_impressions", "Feature", "Consistency signal, not just a raw total."),
    ("days_with_sessions", "Feature", "Consistency signal, not just a raw total."),
    ("impressions_last_30d", "Excluded", "The label is defined FROM last-30 vs prev-30 impressions -- using this would let a model reconstruct the label (leakage)."),
    ("clicks_last_30d", "Excluded", "Same family as impressions_last_30d -- excluded together to avoid partial leakage."),
    ("sessions_last_30d", "Excluded", "Same family as impressions_last_30d -- excluded together."),
    ("impressions_prev_30d", "Excluded", "Same reason as impressions_last_30d -- half of the label's own formula."),
    ("clicks_prev_30d", "Excluded", "Same family, excluded together."),
    ("sessions_prev_30d", "Excluded", "Same family, excluded together."),
    ("ctr", "Feature", "Trailing 90-day rate, describes current state, not derived from the label."),
    ("avg_position", "Feature", "Current ranking position. 0 means 'no data', handled with a has_position_data flag."),
    ("engagement_rate", "Feature", "Trailing 90-day rate."),
    ("scroll_rate", "Feature", "Trailing 90-day rate. Can exceed 100 by definition -- not a bug."),
    ("ai_traffic_pct", "Feature", "Trailing 90-day rate. Can exceed 100 by definition -- not a bug."),
    ("impression_tier", "Excluded", "Redundant restatement of impressions_90d."),
    ("position_tier", "Excluded", "Redundant restatement of avg_position."),
    ("trend_direction", "Label / proxy", "This IS the label (is_declining_label = trend_direction == 'down'). Never a feature."),
    ("trend_pct", "Label / proxy", "Defines trend_direction. Never a feature."),
]

contract_df = pd.DataFrame(field_contract, columns=["field", "bucket", "why"])
print(contract_df["bucket"].value_counts())
contract_df


bucket
Feature          25
Excluded         15
Context           2
Label / proxy     2
Name: count, dtype: int64


,field,bucket,why
0,content_id,Context,"Pseudonym. Row identifier only, never a feature."
1,client_id,Context,Pseudonym. Grouping/splitting key only (client...
2,search_volume,Feature,"Describes the target keyword, known before any..."
3,competition,Feature,"Keyword competition score, known ahead of time."
4,competition_level,Feature,Categorical version of competition.
5,cpc,Feature,"Keyword economics, known ahead of time."
6,content_type,Feature,Structural property of the page.
7,main_intent,Feature,Search-intent category of the page.
8,word_count,Feature,"Content property, known before any performance..."
9,char_count,Feature,"Content property, known before any performance..."


## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim above gets a query here — grain (no duplicate `content_id`), the systematic (not
random) missingness pattern by `content_type`, the `avg_position == 0` / rate-over-100 gotchas
from the data dictionary, and the `trend_pct` blank-fill rule.

In [3]:
# ── Verify the contract claims with queries ──

# Missingness, overall
miss = df.isna().mean().sort_values(ascending=False)
print("Fields with any missing values (share of rows):")
print((miss[miss > 0] * 100).round(1).astype(str) + "%")

# Missingness follows content_type, not randomness -- the trap the skill warns about
print("\nsearch_volume / word_count missing rate, BY content_type:")
print((df.groupby("content_type")[["search_volume", "word_count"]]
       .apply(lambda g: g.isna().mean()) * 100).round(1))

# avg_position == 0 -- confirms the "no data" gotcha, not a real rank
print(f"\navg_position == 0 (means 'no position data', not rank zero): {(df['avg_position'] == 0).sum():,} rows")

# scroll_rate / ai_traffic_pct exceeding 100 -- confirms the dictionary's warning, not a bug
print(f"scroll_rate > 100: {(df['scroll_rate'] > 100).sum():,} rows | "
      f"ai_traffic_pct > 100: {(df['ai_traffic_pct'] > 100).sum():,} rows")

# trend_pct missing -- confirms it's blank when impressions_prev_30d == 0
mismatch = ((df["trend_pct"].isna()) & (df["impressions_prev_30d"] != 0)).sum()
print(f"trend_pct missing: {df['trend_pct'].isna().sum():,} rows | "
      f"of those, rows where impressions_prev_30d != 0 (should be 0 if the rule holds): {mismatch}")


Fields with any missing values (share of rows):
provider_used        71.5%
word_count           25.7%
char_count           25.7%
word_count_tier      25.7%
char_count_tier      25.7%
model_used           19.1%
trend_pct            11.3%
competition_level     8.7%
search_volume         8.2%
cpc                   8.2%
competition           8.2%
main_intent           7.9%
scroll_rate           0.4%
dtype: str

search_volume / word_count missing rate, BY content_type:
                    search_volume  word_count
content_type                                 
comparison article            0.0         0.0
feedly article              100.0         0.0
keyword article               1.4        28.3

avg_position == 0 (means 'no position data', not rank zero): 1,205 rows
scroll_rate > 100: 119 rows | ai_traffic_pct > 100: 23 rows
trend_pct missing: 3,388 rows | of those, rows where impressions_prev_30d != 0 (should be 0 if the rule holds): 0


## 4. Data limits

- **Single snapshot, not a time series.** This file cannot show week-over-week movement in
  `avg_position` or `ctr` — only the derived 30d-vs-30d trend exists, and that trend is exactly
  what the label is built from, so it can't double as a feature (see the leakage notebook).
- **Partial client coverage.** 32 of the warehouse's 104 clients are represented here, and rows
  per client range from 3 to 7,008 — some clients barely have enough rows for a reliable
  per-client conclusion; grouped analysis should watch for that imbalance.
- **`provider_used`/`model_used` are mostly missing** (71.5% / 19.1%) and marked "not a model
  feature" in the data dictionary regardless — they describe how the content was authored, not
  how it's performing, so I don't try to recover them.

In [4]:
# ── One more limit worth confirming: this file is a single snapshot, not a time series ──
# Every row has exactly one impressions_90d value -- there's no way to see how avg_position
# or ctr moved week over week from this file alone; only the derived 30d-vs-30d trend exists.
print("Distinct 'as of' snapshots implied by this file: 1 (all rows share one export window)")
print(f"Clients represented: {df['client_id'].nunique()} of the full warehouse's 104 (per flyrank-data skill)")
print(f"Rows per client -- min: {df.groupby('client_id').size().min()}, "
      f"max: {df.groupby('client_id').size().max()}, "
      f"median: {int(df.groupby('client_id').size().median())}")


Distinct 'as of' snapshots implied by this file: 1 (all rows share one export window)
Clients represented: 32 of the full warehouse's 104 (per flyrank-data skill)
Rows per client -- min: 3, max: 7008, median: 567


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
